In [ ]:
# Cell 1 — Spark session for monitoring (read-only)
from delta import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName('monitor') \
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension') \
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog') \
    .config(
        'spark.jars.packages',
        'io.delta:delta-core_2.12:2.4.0'
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Monitor session ready.')

In [ ]:
# Cell 2 — Latest bronze rows (re-run to refresh)
spark.read.format('delta').load('../delta/bronze/btc_trades') \
    .orderBy(col('event_time').desc()) \
    .limit(20) \
    .show(truncate=False)

In [ ]:
# Cell 3 — Latest silver aggregations (re-run to refresh)
spark.read.format('delta').load('../delta/silver/btc_aggregates') \
    .orderBy(col('window_start').desc()) \
    .limit(10) \
    .show(truncate=False)

In [ ]:
# Cell 4 — Delta table history (audit log)
DeltaTable.forPath(spark, '../delta/bronze/btc_trades') \
    .history() \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

In [ ]:
# Cell 5 — Row count per symbol
spark.read.format('delta').load('../delta/bronze/btc_trades') \
    .groupBy('symbol') \
    .count() \
    .show()